# D1: Monthly Report Generator

---

## Overview

Generate automated monthly housing development reports.

**Report Contents:**
- New proposals
- New approvals
- Construction starts
- Completions

**Output Formats:**
- HTML
- PDF (via HTML)
- JSON (for dashboards)

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent.parent))

from modules.report_generator import (
    generate_monthly_report,
    generate_status_summary,
    export_to_html,
    export_to_json
)
from modules.data_loader import load_csv

# Configuration
with open('../../config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])
REPORTS_DIR = DATA_DIR / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

print(f"Reports directory: {REPORTS_DIR}")

## 2. Load Current Data

In [ ]:
# Load housing projects
housing_path = DATA_DIR / 'housing_projects_FINAL.csv'
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"Total units: {df['net_units'].sum():,.0f}")

## 3. Generate Report Data

In [ ]:
# Generate monthly report
if df is not None:
    report = generate_monthly_report(df, datetime.now())
    
    print(f"Report Month: {report['report_month']}")
    print(f"Generated: {report['generated_at']}")
    print("\nMetrics:")
    for key, value in report['metrics'].items():
        print(f"  {key}: {value}")

## 4. Current Pipeline Summary

In [ ]:
# Status summary
if df is not None:
    summary = generate_status_summary(df)
    
    print("Pipeline Summary:")
    print("="*60)
    print(f"Total Projects: {summary['total_projects']}")
    print(f"Total Units: {summary['total_units']:,}")
    
    print("\nBy Status:")
    for status, count in sorted(summary['by_status'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {status}: {count}")

## 5. Top Projects This Period

In [ ]:
# Largest projects currently in pipeline
if df is not None:
    print("Largest Active Projects:")
    print("="*60)
    
    top_projects = df.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']]
    display(top_projects)

## 6. Export HTML Report

In [ ]:
# Generate HTML report
if df is not None:
    timestamp = datetime.now().strftime('%Y%m')
    html_path = REPORTS_DIR / f'housing_report_{timestamp}.html'
    
    export_to_html(report, html_path)
    print(f"\nHTML report saved: {html_path}")

## 7. Export JSON for Dashboard

In [ ]:
# Export JSON for dashboard
if df is not None:
    json_path = REPORTS_DIR / f'housing_report_{timestamp}.json'
    export_to_json(report, json_path)
    print(f"JSON report saved: {json_path}")

## 8. Report Preview

In [ ]:
# Display report preview
from IPython.display import HTML, display

if html_path.exists():
    with open(html_path) as f:
        html_content = f.read()
    display(HTML(html_content))

---

## Summary

This notebook:
- Generated monthly housing development report
- Created pipeline summary
- Exported to HTML and JSON

**Next:** Run `D2_dashboard_data_export.ipynb` for dashboard data.